In [ ]:
# Cell 1 - Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pickle, json, os, warnings
warnings.filterwarnings('ignore')
print("TensorFlow version:", tf.__version__)

In [ ]:
# Cell 2 - Load Best Model & Artifacts
MODEL_PATH = '../models/hybrid_cnn_lstm_gru.keras'
model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded:", MODEL_PATH)
model.summary()

with open('../data/processed/scaler_y.pkl',    'rb') as f: scaler_y    = pickle.load(f)
with open('../data/processed/encoders.pkl',    'rb') as f: encoders    = pickle.load(f)
with open('../data/processed/top_features.pkl','rb') as f: feature_cols = pickle.load(f)

SEQ_LENGTH = 30
print(f"\nFeatures loaded: {len(feature_cols)}")

In [ ]:
# Cell 3 - Load Full Engineered Data
full_df = pd.read_csv('../data/processed/full_engineered.csv', parse_dates=['Date'])
print("Data loaded:", full_df.shape)
print("Date range:", full_df['Date'].min().date(), "to", full_df['Date'].max().date())

In [ ]:
# Cell 4 - Rolling Forecast Function
def forecast_sku(df, sku_id, feature_cols, model, scaler_y, seq_length=30, n_steps=30):
    """
    Forecast next n_steps days for a given SKU using rolling (auto-regressive) prediction.
    Each step's prediction is fed back as input for the next step.
    """
    sku_data = df[df['SKU_ID'] == sku_id].sort_values('Date').fillna(0)
    if len(sku_data) < seq_length:
        raise ValueError(f"Not enough data for {sku_id}. Need >= {seq_length} rows.")

    history = sku_data[feature_cols].values[-seq_length:].astype(np.float32)
    target_idx = feature_cols.index('Units_Sold_scaled') if 'Units_Sold_scaled' in feature_cols else 0

    predictions, current_window = [], history.copy()
    for _ in range(n_steps):
        inp = current_window[-seq_length:].reshape(1, seq_length, len(feature_cols))
        pred_scaled = model.predict(inp, verbose=0)[0][0]
        pred_actual = scaler_y.inverse_transform([[pred_scaled]])[0][0]
        predictions.append(pred_actual)
        new_row = current_window[-1].copy()
        new_row[target_idx] = pred_scaled
        current_window = np.vstack([current_window[1:], new_row])

    forecast_dates = pd.date_range(
        start=sku_data['Date'].max() + pd.Timedelta(days=1), periods=n_steps)
    return pd.DataFrame({'Date': forecast_dates, 'SKU_ID': sku_id,
                         'Forecasted_Units_Sold': predictions})

In [ ]:
# Cell 5 - Run Inference for Sample SKUs
sample_skus = ['SKU_1', 'SKU_5', 'SKU_10']
all_forecasts = []
for sku in sample_skus:
    fc = forecast_sku(full_df, sku, feature_cols, model, scaler_y,
                      seq_length=SEQ_LENGTH, n_steps=30)
    all_forecasts.append(fc)
    print(f"\n{sku} - Next 30 Days Forecast (first 5 rows):")
    print(fc[['Date', 'Forecasted_Units_Sold']].head().to_string(index=False))

combined_forecast = pd.concat(all_forecasts, ignore_index=True)
print(f"\nTotal forecast rows: {len(combined_forecast)}")

In [ ]:
# Cell 6 - Forecast Visualization
os.makedirs('../outputs/plots', exist_ok=True)
fig, axes = plt.subplots(len(sample_skus), 1, figsize=(14, 5 * len(sample_skus)))
for idx, sku in enumerate(sample_skus):
    hist = full_df[full_df['SKU_ID'] == sku].sort_values('Date').tail(60)
    fc   = combined_forecast[combined_forecast['SKU_ID'] == sku]
    axes[idx].plot(hist['Date'], hist['Units_Sold'], color='steelblue', linewidth=1.5, label='Historical')
    axes[idx].plot(fc['Date'], fc['Forecasted_Units_Sold'], color='coral', linewidth=2,
                   linestyle='--', marker='o', markersize=4, label='30-Day Forecast')
    axes[idx].axvline(x=hist['Date'].max(), color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
    axes[idx].set_title(f'{sku} - 30-Day Demand Forecast', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Date'); axes[idx].set_ylabel('Units Sold')
    axes[idx].legend(); axes[idx].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../outputs/plots/08_demand_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 7 - Reorder Alert System
def generate_reorder_alerts(df, forecast_df, safety_stock_multiplier=1.5):
    """Generate inventory reorder alerts based on forecast vs current inventory."""
    alerts = []
    for sku in forecast_df['SKU_ID'].unique():
        fc          = forecast_df[forecast_df['SKU_ID'] == sku]
        avg_fc      = fc['Forecasted_Units_Sold'].mean()
        sku_hist    = df[df['SKU_ID'] == sku]
        curr_inv    = sku_hist['Inventory_Level'].iloc[-1]
        reorder_pt  = sku_hist['Reorder_Point'].iloc[-1]
        lead_time   = sku_hist['Supplier_Lead_Time_Days'].iloc[-1]
        safety_stk  = avg_fc * lead_time * 0.1 * safety_stock_multiplier
        status      = ('CRITICAL' if curr_inv < reorder_pt
                       else 'WARNING' if curr_inv < reorder_pt * 1.5
                       else 'OK')
        alerts.append({'SKU_ID': sku,
                       'Current_Inventory': round(curr_inv, 0),
                       'Reorder_Point': round(reorder_pt, 0),
                       'Avg_Forecast_Demand': round(avg_fc, 2),
                       'Safety_Stock': round(safety_stk, 2),
                       'Lead_Time_Days': lead_time,
                       'Status': status})
    return pd.DataFrame(alerts).set_index('SKU_ID')

alerts = generate_reorder_alerts(full_df, combined_forecast)
print("\n REORDER ALERT SYSTEM")
print("=" * 65)
print(alerts.to_string())

In [ ]:
# Cell 8 - Save Forecasts & Alerts
os.makedirs('../outputs/reports', exist_ok=True)
combined_forecast.to_csv('../outputs/reports/demand_forecast_30days.csv', index=False)
alerts.to_csv('../outputs/reports/reorder_alerts.csv')
print("\n✅ Deployment & Inference Complete!")
print("  Forecasts : outputs/reports/demand_forecast_30days.csv")
print("  Alerts    : outputs/reports/reorder_alerts.csv")
print("  Plot      : outputs/plots/08_demand_forecast.png")

In [ ]:
# Cell 9 - Production Forecaster Class (API-ready)
class SupplyChainForecaster:
    """
    Production-ready wrapper for supply chain demand forecasting.
    Easily served via Flask, FastAPI, or cloud functions.
    
    Usage:
        forecaster = SupplyChainForecaster(model_path, scaler_path, features_path)
        predictions = forecaster.predict(sku_history_df, n_steps=7)
    """
    def __init__(self, model_path, scaler_path, features_path):
        self.model = tf.keras.models.load_model(model_path)
        with open(scaler_path,   'rb') as f: self.scaler_y    = pickle.load(f)
        with open(features_path, 'rb') as f: self.feature_cols = pickle.load(f)
        self.seq_length = 30
        print(f"✅ Forecaster initialized | Features: {len(self.feature_cols)}")

    def predict(self, sku_history_df, n_steps=7):
        """Predict the next n_steps for a given SKU dataframe."""
        X = sku_history_df[self.feature_cols].fillna(0).values.astype(np.float32)
        if len(X) < self.seq_length:
            raise ValueError(f"Need >= {self.seq_length} rows of history.")
        preds, window = [], X[-self.seq_length:].copy()
        for _ in range(n_steps):
            inp = window.reshape(1, self.seq_length, -1)
            p_s = self.model.predict(inp, verbose=0)[0][0]
            p_a = self.scaler_y.inverse_transform([[p_s]])[0][0]
            preds.append(p_a)
            new_row = window[-1].copy()
            window = np.vstack([window[1:], new_row])
        return preds

# --- Instantiate & quick test ---
forecaster = SupplyChainForecaster(
    model_path    = '../models/hybrid_cnn_lstm_gru.keras',
    scaler_path   = '../data/processed/scaler_y.pkl',
    features_path = '../data/processed/top_features.pkl'
)
sku_data = full_df[full_df['SKU_ID'] == 'SKU_1'].sort_values('Date')
preds_7  = forecaster.predict(sku_data, n_steps=7)
print("\n7-Day forecast for SKU_1:", [round(p, 2) for p in preds_7])